# Point Estimation And Confidence Intervals

**Official MA1001B Alignment:** *5.1 point estimation; 5.2 intervals; 5.3 means; 5.4 standard error; 5.5 proportions.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Calculate point estimates and standard errors for continuous means and binary proportions.
- Construct parametric 95% confidence intervals for means using the t-distribution.
- Construct normal-approximation 95% confidence intervals for binary proportions (`p_hat +/- z*SE`).
- Interpret confidence interval widths and confidence levels correctly in stakeholder communication.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model estimation uncertainty to establish plausible bounds for unknown population parameters.
- **2. Computational Link (How Python represents it):** We combine SciPy interval methods (`stats.t.interval`) with NumPy standard error formulas (`np.sqrt(p*(1-p)/n)`).
- **3. Decision Link (How it guides action):** Interval estimates prevent executives from making false policy commitments based on imprecise point averages.


## Decision Scenario

> **The Problem:** A survey team needs to estimate average satisfaction and support for a policy. A point estimate alone is too precise for decision making.


## Conceptual Explanation

Estimation separates what the sample says from what we infer about the population. A confidence interval is a procedure that, under its assumptions, captures the true parameter at a stated long-run rate. It is not a probability statement about one fixed parameter after the interval is computed.


## Mathematical Anchor

A common interval structure is estimate +/- critical value times standard error.


## Data And Workflow Notes

Uses simulated survey data with one numeric score and one binary support variable.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Survey Data Simulation & Inspection

We generate a simulated survey dataset of n=180 respondents containing a continuous satisfaction score (0-100) and a binary policy support indicator (True/False).


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Simulate survey responses for n=180 citizens
survey = pd.DataFrame({
    "satisfaction_score": rng.normal(loc=72, scale=12, size=180).clip(0, 100),
    "supports_policy": rng.binomial(n=1, p=0.58, size=180),  # 58% true population support
})
survey.head()


### Step 2: Confidence Interval for a Continuous Mean

We compute the sample mean satisfaction score, its standard error, and the exact 95% Student's t-distribution confidence interval.


In [ ]:
# Estimate continuous population mean satisfaction
mean_est = survey["satisfaction_score"].mean()
mean_se = survey["satisfaction_score"].std(ddof=1) / np.sqrt(len(survey))
mean_ci = stats.t.interval(
    confidence=0.95,
    df=len(survey) - 1,
    loc=mean_est,
    scale=mean_se,
)

pd.Series({
    "point_estimate_mean": mean_est,
    "standard_error": mean_se,
    "95%_CI_lower": mean_ci[0],
    "95%_CI_upper": mean_ci[1],
    "margin_of_error": (mean_ci[1] - mean_ci[0]) / 2
}).round(3)


### Step 3: Confidence Interval for a Binary Proportion

We compute the sample proportion of policy supporters, calculate the proportion standard error `sqrt(p*(1-p)/n)`, and derive the 95% normal-approximation interval (`z=1.96`).


In [ ]:
# Estimate binary population proportion supporting policy
p_hat = survey["supports_policy"].mean()
n = len(survey)
prop_se = np.sqrt(p_hat * (1 - p_hat) / n)
prop_ci = (p_hat - 1.96 * prop_se, p_hat + 1.96 * prop_se)

pd.Series({
    "point_estimate_proportion": p_hat,
    "proportion_standard_error": prop_se,
    "95%_CI_lower": prop_ci[0],
    "95%_CI_upper": prop_ci[1],
    "margin_of_error": 1.96 * prop_se
}).round(4)


### Step 4: Verifying Proportion Interval via Bootstrap

We draw 1,000 bootstrap resamples of the binary support column to check if the non-parametric bootstrap percentile interval matches our normal approximation.


In [ ]:
# Verify proportion confidence bounds using 1,000 bootstrap resamples
bootstrap_support = [
    survey.sample(n=len(survey), replace=True, random_state=seed)["supports_policy"].mean()
    for seed in range(1000)
]

bootstrap_ci = pd.Series(bootstrap_support).quantile([0.025, 0.50, 0.975])
pd.DataFrame({
    "Normal_Approximation_CI": [prop_ci[0], p_hat, prop_ci[1]],
    "Bootstrap_Percentile_CI": bootstrap_ci.values
}, index=["Lower_2.5%", "Point_Estimate", "Upper_97.5%"]).round(4)


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Would you claim that a definitive majority of the population supports the policy? Justify your answer using the confidence interval, not just the sample proportion.

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Saying 'there is a 95% probability that the true parameter lies in this specific numerical interval' (the parameter is fixed; confidence is in the long-run procedure).
- **Warning:** Reporting a numerical confidence interval without explicitly stating the confidence level (e.g., 90%, 95%, 99%).
- **Warning:** Ignoring whether the underlying survey sample was collected via random sampling versus biased convenience sampling.


## Independent Practice

> [!TIP]
> **Your Task:**
> Recalculate the satisfaction confidence interval using simulated sample sizes of `n=60` and `n=600`. Explain how the margin of error changes as sample size scales.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** What is the fundamental difference between a point estimate and a confidence interval estimate?

*Write your brief conceptual reflection below:*
